In [1]:
import random
from typing import List, Dict, Tuple
import chromadb
from chromadb.utils import embedding_functions
from dataclasses import dataclass
import json


In [2]:
# MBTI 질문 데이터
MBTI_QUESTIONS = [
    # E vs I 관련 질문
    "친구들과 파티에 가는 것을 즐기나요?",
    "새로운 사람들과 만나는 것이 즐겁나요?",
    "혼자만의 시간이 필요한가요?",
    "대화를 시작하는 것을 좋아하나요?",
    "조용한 장소보다 활기찬 장소를 선호하나요?",
    
    # S vs N 관련 질문
    "세부사항에 주목하는 편인가요?",
    "미래의 가능성을 상상하는 것을 좋아하나요?",
    "현실적인 해결책을 선호하나요?",
    "직관적인 느낌을 중요하게 생각하나요?",
    "새로운 아이디어를 탐구하는 것을 좋아하나요?",
    
    # T vs F 관련 질문
    "논리적인 분석을 중요하게 생각하나요?",
    "다른 사람의 감정을 쉽게 공감하나요?",
    "객관적인 사실을 중시하나요?",
    "결정할 때 감정을 고려하나요?",
    "갈등 상황에서 공정성을 중요시하나요?",
    
    # J vs P 관련 질문
    "계획을 세우고 따르는 것을 좋아하나요?",
    "즉흥적인 결정을 자주 하나요?",
    "마감기한을 잘 지키나요?",
    "유연한 상황 대처를 선호하나요?",
    "체계적인 방식으로 일하는 것을 좋아하나요?"   
]

In [3]:
# 감정 상태 분석을 위한 질문 템플릿
EMOTION_QUESTIONS = {
    "INTJ": "혼자만의 시간에 어떤 감정을 느끼시나요?",
    "INTP": "새로운 아이디어를 탐구할 때 어떤 기분이 드나요?",
    "INFJ": "다른 사람의 감정을 이해할 때 본인의 감정은 어떤가요?",
    "INFP": "창의적인 활동을 할 때 어떤 감정을 느끼시나요?",
    "ENTJ": "목표를 달성하기 위해 노력할 때 어떤 감정을 느끼시나요?",
    "ENTP": "토론을 할 때 자신의 감정은 어떤가요?",
    "ENFJ": "다른 사람을 도울 때 본인은 어떤 기분이 드나요?",
    "ENFP": "새로운 사람을 만날 때 어떤 감정을 느끼시나요?",
    "ISTJ": "규칙이나 절차를 따를 때 어떤 기분이 드나요?",
    "ISFJ": "친구를 위해 헌신할 때 어떤 감정을 느끼시나요?",
    "ISTP": "문제를 해결할 때 어떤 감정을 느끼시나요?",
    "ISFP": "자연 속에서 시간을 보낼 때 어떤 기분이 드나요?",
    "ESTP": "즉흥적인 결정을 내릴 때 어떤 감정을 느끼시나요?",
    "ESFP": "사람들과 함께 할 때 어떤 기분이 드나요?",
    "ESTJ": "조직을 이끌 때 어떤 감정을 느끼시나요?",
    "ESFJ": "사람들과의 관계를 유지할 때 어떤 기분이 드나요?"
}

In [4]:
@dataclass
class MBTIAnalysis:
    mbti_type: str
    confidence: float
    primary_traits: Dict[str, float]

@dataclass
class EmotionAnalysis:
    current_emotion: str
    intensity: float
    contributing_factors: List[str]

In [5]:
class MBTIAgent:
    def __init__(self):
        self.chroma_client = chromadb.Client()
        self.embedding_function = embedding_functions.DefaultEmbeddingFunction()
        self.setup_database()

    def setup_database(self):
        """ChromaDB에 MBTI 질문 저장"""
        self.collection = self.chroma_client.create_collection(
            name="mbti_questions",
            embedding_function=self.embedding_function
        )
        
        # 질문들을 DB에 저장
        self.collection.add(
            documents=MBTI_QUESTIONS,
            ids=[f"q_{i}" for i in range(len(MBTI_QUESTIONS))]
        )

    def get_random_questions(self, n: int = 4) -> str:
        """랜덤하게 n개의 질문 선택"""
        selected_ids = random.sample([f"q_{i}" for i in range(len(MBTI_QUESTIONS))], n)
        questions = self.collection.get(ids=selected_ids)
        return " ".join([f"{i+1}. {q}" for i, q in enumerate(questions['documents'])])

    def analyze_response(self, response: str) -> MBTIAnalysis:
        """사용자 응답을 분석하여 MBTI 결정"""
        # 실제 구현에서는 더 복잡한 분석 로직이 필요
        # 예시로 간단한 결과만 반환
        return MBTIAnalysis(
            mbti_type="INTJ",
            confidence=0.85,
            primary_traits={
                "I": 0.7,
                "N": 0.8,
                "T": 0.9,
                "J": 0.6
            }
        )



In [6]:
class EmotionAgent:
    def __init__(self):
        self.emotion_questions = EMOTION_QUESTIONS

    def get_emotion_question(self, mbti_type: str) -> str:
        """MBTI 유형에 따른 감정 관련 질문 생성"""
        return self.emotion_questions.get(mbti_type, "현재 기분이 어떠신가요?")

    def analyze_emotion(self, response: str) -> EmotionAnalysis:
        """감정 상태 분석"""
        # 실제 구현에서는 감정 분석 모델 사용
        return EmotionAnalysis(
            current_emotion="peaceful",
            intensity=0.7,
            contributing_factors=["positive environment", "achievement"]
        )

In [7]:
class MovieRecommendationAgent:
    def get_movie_preferences(self, mbti: MBTIAnalysis, emotion: EmotionAnalysis) -> Dict:
        """MBTI와 감정 상태를 기반으로 영화 취향 분석"""
        preferences = {
            "genres": self._get_preferred_genres(mbti, emotion),
            "themes": self._get_preferred_themes(mbti, emotion),
            "mood": self._get_preferred_mood(emotion),
            "complexity": self._get_story_complexity(mbti)
        }
        return preferences

    def _get_preferred_genres(self, mbti: MBTIAnalysis, emotion: EmotionAnalysis) -> List[str]:
        genres = []
        if mbti.mbti_type.startswith("IN"):
            genres.extend(["sci-fi", "mystery", "psychological thriller"])
        elif mbti.mbti_type.startswith("ES"):
            genres.extend(["action", "adventure", "comedy"])
        return genres

    def _get_preferred_themes(self, mbti: MBTIAnalysis, emotion: EmotionAnalysis) -> List[str]:
        if "T" in mbti.mbti_type:
            return ["complex relationships", "philosophical questions"]
        return ["personal growth", "emotional journey"]

    def _get_preferred_mood(self, emotion: EmotionAnalysis) -> str:
        # 현재 감정과 반대되는 무드 또는 보완적인 무드 추천
        mood_mapping = {
            "peaceful": "inspiring",
            "stressed": "calming",
            "excited": "thought-provoking",
            "sad": "uplifting"
        }
        return mood_mapping.get(emotion.current_emotion, "balanced")

    def _get_story_complexity(self, mbti: MBTIAnalysis) -> str:
        if "N" in mbti.mbti_type and "T" in mbti.mbti_type:
            return "complex"
        return "moderate"

In [8]:
class PersonalityBasedRecommendationSystem:
    def __init__(self):
        self.mbti_agent = MBTIAgent()
        self.emotion_agent = EmotionAgent()
        self.movie_agent = MovieRecommendationAgent()

    def start_analysis(self):
        # 1. MBTI 분석
        mbti_questions = self.mbti_agent.get_random_questions()
        print("다음 질문들에 답해주세요:")
        print(mbti_questions)
        
        # 실제 구현에서는 사용자 입력 받기
        user_response = input("답변을 입력해주세요: ")
        mbti_result = self.mbti_agent.analyze_response(user_response)

        # 2. 감정 상태 분석
        emotion_question = self.emotion_agent.get_emotion_question(mbti_result.mbti_type)
        print("\n감정 상태 분석을 위한 질문:")
        print(emotion_question)
        
        emotion_response = input("답변을 입력해주세요: ")
        emotion_result = self.emotion_agent.analyze_emotion(emotion_response)

        # 3. 영화 취향 분석
        movie_preferences = self.movie_agent.get_movie_preferences(mbti_result, emotion_result)

        return {
            "mbti_analysis": mbti_result,
            "emotion_analysis": emotion_result,
            "movie_preferences": movie_preferences
        }

In [9]:
system = PersonalityBasedRecommendationSystem()

In [10]:
results = system.start_analysis()

다음 질문들에 답해주세요:
1. 새로운 사람들과 만나는 것이 즐겁나요? 2. 대화를 시작하는 것을 좋아하나요? 3. 현실적인 해결책을 선호하나요? 4. 직관적인 느낌을 중요하게 생각하나요?

감정 상태 분석을 위한 질문:
혼자만의 시간에 어떤 감정을 느끼시나요?


In [11]:
results

{'mbti_analysis': MBTIAnalysis(mbti_type='INTJ', confidence=0.85, primary_traits={'I': 0.7, 'N': 0.8, 'T': 0.9, 'J': 0.6}),
 'emotion_analysis': EmotionAnalysis(current_emotion='peaceful', intensity=0.7, contributing_factors=['positive environment', 'achievement']),
 'movie_preferences': {'genres': ['sci-fi',
   'mystery',
   'psychological thriller'],
  'themes': ['complex relationships', 'philosophical questions'],
  'mood': 'inspiring',
  'complexity': 'complex'}}

In [ ]:
results1 = system.start_analysis()
print(results1)

In [ ]:
# 여러개 서술이 카톡에서 진행할때는 어려울 것 같다
# VS 게임으로